# NovaCart — Silver Category Translation Transformation

## 1. Import Libraries

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 2. Define Storage Paths

In [0]:
BRONZE_CATEGORY_TRANSLATION_PATH = (
    "abfss://bronze@stnovacartdev.dfs.core.windows.net/"
    "olist/category_translation"
)

SILVER_CATEGORY_TRANSLATION_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/category_translation"
)

QUARANTINE_CATEGORY_TRANSLATION_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist/category_translation"
)

print(f"Bronze path: {BRONZE_CATEGORY_TRANSLATION_PATH}")
print(f"Silver path: {SILVER_CATEGORY_TRANSLATION_PATH}")
print(f"Quarantine path: {QUARANTINE_CATEGORY_TRANSLATION_PATH}")

## 3. Read Bronze Category Translation Data

In [0]:
category_translation_bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_CATEGORY_TRANSLATION_PATH)
)

bronze_row_count = category_translation_bronze_df.count()

print("Bronze category translation loaded successfully.")
print(f"Bronze row count: {bronze_row_count}")

category_translation_bronze_df.printSchema()
display(category_translation_bronze_df.limit(10))

## 4. Validate Required Columns

In [0]:
required_columns = [
    "product_category_name",
    "product_category_name_english",
    "_source_file",
    "_ingestion_timestamp",
    "_batch_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in category_translation_bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Bronze columns: {missing_columns}"
    )

print("Required-column validation passed.")

## 5. Profile Missing and Invalid Category Translation Values

In [0]:
category_translation_profile_df = category_translation_bronze_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        (
            F.col("product_category_name").isNull()
            | (F.trim(F.col("product_category_name")) == "")
        ).cast("int")
    ).alias("invalid_product_category_name"),

    F.sum(
        (
            F.col("product_category_name_english").isNull()
            | (F.trim(F.col("product_category_name_english")) == "")
        ).cast("int")
    ).alias("invalid_product_category_name_english"),
)

display(category_translation_profile_df)

## 6. Print Full Category Translation Quality Profile

In [0]:
profile = category_translation_profile_df.first().asDict()

for metric, value in profile.items():
    print(f"{metric}: {value}")

## 7. Check Duplicate Portuguese Category Names

In [0]:
duplicate_category_names_df = (
    category_translation_bronze_df
    .groupBy("product_category_name")
    .count()
    .filter(
        F.col("product_category_name").isNotNull()
        & (F.col("count") > 1)
    )
)

duplicate_category_name_count = duplicate_category_names_df.count()

print(
    "Number of product_category_name values appearing more than once: "
    f"{duplicate_category_name_count}"
)

display(duplicate_category_names_df.limit(20))

## 8. Check Exact Duplicate Records

In [0]:
business_columns = [
    "product_category_name",
    "product_category_name_english",
]

exact_duplicate_count = (
    bronze_row_count
    - category_translation_bronze_df
        .dropDuplicates(business_columns)
        .count()
)

print(f"Exact duplicate category translation rows: {exact_duplicate_count}")

## 9. Clean and Standardize Category Translation Fields

In [0]:
category_translation_cleaned_df = (
    category_translation_bronze_df
    .withColumn(
        "product_category_name",
        F.lower(F.trim(F.col("product_category_name")))
    )
    .withColumn(
        "product_category_name_english",
        F.lower(F.trim(F.col("product_category_name_english")))
    )
)

In [0]:
category_key_window = Window.partitionBy(
    "product_category_name"
)

category_translation_checked_df = (
    category_translation_cleaned_df
    .withColumn(
        "_duplicate_key_count",
        F.count("*").over(category_key_window)
    )
)

## 10. Define Category Translation Validation Rules

In [0]:
invalid_product_category_name_condition = (
    F.col("product_category_name").isNull()
    | (F.col("product_category_name") == "")
)

invalid_product_category_name_english_condition = (
    F.col("product_category_name_english").isNull()
    | (F.col("product_category_name_english") == "")
)

## 11. Assign Category Translation Rejection Reasons

In [0]:
category_translation_validated_df = (
    category_translation_checked_df
    .withColumn(
        "_rejection_reason",

        F.when(
            invalid_product_category_name_condition,
            F.lit("MISSING_PRODUCT_CATEGORY_NAME")
        )
        .when(
            invalid_product_category_name_english_condition,
            F.lit("MISSING_PRODUCT_CATEGORY_NAME_ENGLISH")
        )
        .when(
            F.col("_duplicate_key_count") > 1,
            F.lit("DUPLICATE_PRODUCT_CATEGORY_NAME")
        )
        .otherwise(F.lit(None))
    )
)

## 12. Review Category Translation Validation Results

In [0]:
display(
    category_translation_validated_df
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

## 13. Split Valid and Invalid Category Translation Records

In [0]:
category_translation_valid_df = (
    category_translation_validated_df
    .filter(F.col("_rejection_reason").isNull())
    .drop(
        "_rejection_reason",
        "_duplicate_key_count"
    )
)

category_translation_quarantine_df = (
    category_translation_validated_df
    .filter(F.col("_rejection_reason").isNotNull())
    .drop("_duplicate_key_count")
)

## 14. Add Silver Processing Metadata

In [0]:
category_translation_silver_df = (
    category_translation_valid_df
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

## 15. Add Quarantine Metadata

In [0]:
category_translation_quarantine_df = (
    category_translation_quarantine_df
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_dataset",
        F.lit("category_translation")
    )
)

## 16. Count Silver and Quarantine Records

In [0]:
valid_row_count = category_translation_silver_df.count()
quarantine_row_count = category_translation_quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")
print(f"Bronze input rows: {bronze_row_count}")

## 17. Validate Row-Count Reconciliation

In [0]:
if valid_row_count + quarantine_row_count != bronze_row_count:
    raise ValueError(
        "Row-count validation failed: "
        "Silver rows + quarantine rows do not equal Bronze input rows."
    )

print("Row-count validation passed.")

## 18. Write Valid Category Translation Records to Silver

In [0]:
(
    category_translation_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_CATEGORY_TRANSLATION_PATH)
)

print("Silver category translation written successfully.")

## 19. Write Invalid Category Translation Records to Quarantine

In [0]:
(
    category_translation_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_CATEGORY_TRANSLATION_PATH)
)

print("Category translation quarantine output written successfully.")

## 20. Read Written Delta Outputs

In [0]:
category_translation_silver_written_df = (
    spark.read
    .format("delta")
    .load(SILVER_CATEGORY_TRANSLATION_PATH)
)

category_translation_quarantine_written_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_CATEGORY_TRANSLATION_PATH)
)

silver_written_count = category_translation_silver_written_df.count()
quarantine_written_count = category_translation_quarantine_written_df.count()

print(f"Written Silver rows: {silver_written_count}")
print(f"Written quarantine rows: {quarantine_written_count}")

## 21. Validate Written Outputs

In [0]:
if silver_written_count != valid_row_count:
    raise ValueError(
        "Silver write validation failed: "
        f"expected {valid_row_count}, wrote {silver_written_count}."
    )

if quarantine_written_count != quarantine_row_count:
    raise ValueError(
        "Quarantine write validation failed: "
        f"expected {quarantine_row_count}, wrote "
        f"{quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != bronze_row_count:
    raise ValueError(
        "Final reconciliation failed: "
        "Silver + quarantine does not equal Bronze."
    )

print("Silver category translation pipeline completed successfully.")
print("Final row-count validation passed.")

## 22. Inspect Final Silver Category Translation Dataset

In [0]:
category_translation_silver_written_df.printSchema()

display(
    category_translation_silver_written_df.select(
        "product_category_name",
        "product_category_name_english",
        "_silver_processed_at"
    ).limit(20)
)